# ACIC22 Track-2 reproducer

This notebook reproduces the ACIC22 Track-2 result on a fixed, seeded sample of the challenge's semi-synthetic cohorts, using three published, non-proprietary estimators: cross-fitted **AIPW**, **DR-ATT** with an overlap trim, and a **Rosenbaum** sensitivity bound.

Each cohort is a panel of ~500 practices over four years; the scored estimand is the sample average treatment effect on the treated (SATT). We reduce each cohort to a cross-sectional `(X, T, Y)` problem with a practice-level difference-in-differences change score, estimate the SATT, and aggregate accuracy and calibration across cohorts.

The full internal system adds proprietary methods and reaches tighter numbers; those figures are recorded under `internal_reference` in `expected_results.json` and are out of scope here.

## 1. Setup

Import the estimators and loader. The data must already be on disk (`python data/acic22/fetch.py`); set `ACIC22_DATA_ROOT` if it lives outside `data/acic22/`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from src import (
    aipw_att, dr_att, rosenbaum_gamma_zero,
    load_track2_cohort, enumerate_track2_cohorts,
)
import run

SEED = 42
print('cohorts available:', len(enumerate_track2_cohorts()))

## 2. Load a single cohort

The loader returns the reduced covariate matrix `X`, the treatment indicator `T`, the change-score outcome `Y`, and the true SATT.

In [ ]:
cohort = load_track2_cohort(1)
print('X shape       :', cohort.X.shape)
print('treated / n   :', int(cohort.T.sum()), '/', cohort.n)
print('features      :', cohort.feature_names)
print('true SATT     :', round(cohort.true_ate, 3))

## 3. AIPW walkthrough

Cross-fitted augmented inverse-probability weighting for the effect on the treated. It reports a point estimate, an influence-function standard error, a 95% interval, and the propensity range as an overlap diagnostic.

In [ ]:
a = aipw_att(cohort.X, cohort.T, cohort.Y, random_state=SEED)
print(f'AIPW ATT     : {a.att:+.3f}  (95% CI {a.ci_lo_95:+.3f}, {a.ci_hi_95:+.3f})')
print(f'SE           : {a.se:.3f}')
print(f'propensity   : [{a.propensity_range[0]:.2f}, {a.propensity_range[1]:.2f}]')
print(f'true SATT    : {cohort.true_ate:+.3f}')

## 4. DR-ATT walkthrough

The doubly-robust ATT trims units outside the overlap region before forming the contrast; `n_trimmed` reports how many were dropped.

In [ ]:
d = dr_att(cohort.X, cohort.T, cohort.Y, random_state=SEED)
print(f'DR-ATT       : {d.att:+.3f}  (95% CI {d.ci_lo_95:+.3f}, {d.ci_hi_95:+.3f})')
print(f'SE           : {d.se:.3f}')
print(f'trimmed      : {d.n_trimmed} of {cohort.n}')
print(f'true SATT    : {cohort.true_ate:+.3f}')

## 5. Rosenbaum Gamma-bound

As the assumed confounder strength Gamma grows, the one-sided confidence bound on the effect widens. `gamma_zero` is the strength at which it reaches zero — larger means more robust.

In [ ]:
r = rosenbaum_gamma_zero(d.att, d.se)
print('gamma_zero   :', round(r.gamma_zero, 3), '|', r.note)

plt.figure(figsize=(6, 4))
plt.plot(r.gamma_grid_evaluated, r.lower_bound_grid, marker='.')
plt.axhline(0.0, color='crimson', lw=1, ls='--')
plt.xlabel('Gamma (confounder strength)')
plt.ylabel('one-sided lower bound on |effect|')
plt.title('Rosenbaum widening bound')
plt.tight_layout()
plt.show()

## 6. Aggregate over many cohorts

Run the full pipeline over a seeded sample and aggregate bias, RMSE, coverage, and interval width. (Use `run.select_cohorts(100, SEED)` for the canonical sample; a smaller sample keeps the notebook quick.)

In [ ]:
ids = run.select_cohorts(25, seed=SEED)
results = run.run_reproducer(ids, ('aipw', 'dr_att', 'rosenbaum'), seed=SEED)
print(json.dumps(results, indent=2))

## 7. Compare to the pinned contract

`expected_results.json` pins the canonical 100-cohort targets. The table below shows the canonical targets alongside the internal reference figures.

In [ ]:
expected = json.loads(Path('expected_results.json').read_text())
targets = expected['reproducer_targets']
for method in ('aipw', 'dr_att'):
    t = targets[method]
    print(f"{method:8s} bias={t['bias_mean']['target']:6.2f} "
          f"rmse={t['rmse']['target']:6.2f} "
          f"cov95={t['coverage_95']['target']:.2f} "
          f"width={t['ci_width_mean']['target']:6.2f}")
print('rosenbaum gamma_zero median target:', targets['rosenbaum']['gamma_zero_median']['target'])

## 8. Interpretation

The SATT is small relative to the year-to-year variation in the change score, so per-cohort estimates are noisy and the 95% intervals under-cover. DR-ATT, which restricts to the overlap region, is the more accurate and better-calibrated arm. These are the honest numbers three public doubly-robust methods produce on the Track-2 panels.

## 9. Platform reference

The full internal system — which adds an instrumental-variables estimator, a deep counterfactual estimator, and a paired Rosenbaum inversion — reaches tighter accuracy and calibration than the three public methods shown here. Those figures are recorded under `internal_reference` in `expected_results.json`; the extensions are commercial. Contact the maintainers for a platform demonstration.